In [30]:
import pandas as pd 
from dateutil import parser
import numpy as np

In [31]:
df= pd.read_csv('/Users/dantezamora/Downloads/messy_leads_mock_dataset.csv') 

df2= df.copy()

In [32]:
date_columns = ['created_date', 'first_contact_date', 'conversion_date']

def parse_mixed_date(val) : 
    if pd.isna(val) or str(val).strip() == "":
        return None 
    try: 
        return parser.parse(str(val)).strftime('%Y-%m-%d')
    except(ValueError, TypeError):
        return None
    
for col in date_columns: 
    df2[col] = df2[col].apply(parse_mixed_date)
    
df2.head()

,lead_id,lead_first_name,lead_last_name,job_title,company_name,industry,company_size,city,state,country,...,created_date,first_contact_date,conversion_date,annual_revenue_usd,employees,estimated_deal_value_usd,sales_rep,last_activity,marketing_opt_in,notes
0,L-10019,James,MOORE,Founder,Silverstone Automotive,automotive,501-1000,COLUMBUS,oh,US,...,2026-08-12,2026-08-22,None,500000000,3200,15000,Alex Kim,Phone Call,NaN,Downloaded whitepaper
1,L-10046,ANDREW,JACKSON,Sales Director,Lighthouse Biotech,BIOTECHNOLOGY,1-50,CAMBRIDGE,ma,United States of America,...,2026-08-30,2026-09-11,None,150000000,8500,40000,Chris Brown,Form Submission,NaN,Asked for pricing
2,L-10048,Christopher,WILLIAMS,Product Marketing Manager,Vertex Software,TECHNOLOGY,5000+,Austin,TX,usa,...,2026-06-21,2026-06-23,None,75000000,120,75000,Unassigned,Form Submission,NaN,Referral from partner
3,L-10090,ANDREW,BROWN,Data Analytics Manager,WESTBRIDGE LEGAL,legal,1001-5000,Denver,CO,USA,...,2026-05-16,2026-05-24,None,25000000,45,75000,Jordan Lee,Demo Requested,Yes,Duplicate suspected
4,L-10005,Emily,garcia,RevOps Manager,Metro Hospitality,hospitality,5000+,Miami,FL,US,...,2026-07-15,2026-07-19,None,75000000,12,5000,Chris Brown,Phone Call,no,Referral from partner


In [33]:
#Date Checks 
temp_df = df2[['created_date', 'first_contact_date', 'conversion_date']]
temp_df

,created_date,first_contact_date,conversion_date
0,2026-08-12,2026-08-22,None
1,2026-08-30,2026-09-11,None
2,2026-06-21,2026-06-23,None
3,2026-05-16,2026-05-24,None
4,2026-07-15,2026-07-19,None
...,...,...,...
118,2026-02-02,None,None
119,2026-03-21,2026-03-27,None
120,2026-08-14,2026-08-17,2026-09-10
121,2026-08-06,2026-08-09,None


In [34]:
#String Updates (Casing, Grouping, etc.)

string_columns = df2.select_dtypes(include= 'object').columns 

for col in string_columns: 
    df2[col] = df2[col].apply(
        lambda x: " ".join(str(x).split()).title() if pd.notna(x) else x 
    )

df2['state'] = df2['state'].apply(
    lambda x: x.upper() if pd.notna(x) else x 
)

df2['email'] = df2['email'].apply(
    lambda x: x.lower() if pd.notna(x) else x
)

df2['country'] = df['country'].astype(str).str.strip().str.lower()
df2['country'] = df2['country'].replace(['us', 'usa', 'united states', 'united states of america'], 'United Sates') 

df2['lead_full_name'] = df2['lead_first_name'].fillna('').astype(str) + " " + df2['lead_last_name'].fillna('').astype(str)

df2.head()

,lead_id,lead_first_name,lead_last_name,job_title,company_name,industry,company_size,city,state,country,...,first_contact_date,conversion_date,annual_revenue_usd,employees,estimated_deal_value_usd,sales_rep,last_activity,marketing_opt_in,notes,lead_full_name
0,L-10019,James,Moore,Founder,Silverstone Automotive,Automotive,501-1000,Columbus,OH,United Sates,...,2026-08-22,None,500000000,3200,15000,Alex Kim,Phone Call,NaN,Downloaded Whitepaper,James Moore
1,L-10046,Andrew,Jackson,Sales Director,Lighthouse Biotech,Biotechnology,1-50,Cambridge,MA,United Sates,...,2026-09-11,None,150000000,8500,40000,Chris Brown,Form Submission,NaN,Asked For Pricing,Andrew Jackson
2,L-10048,Christopher,Williams,Product Marketing Manager,Vertex Software,Technology,5000+,Austin,TX,United Sates,...,2026-06-23,None,75000000,120,75000,Unassigned,Form Submission,NaN,Referral From Partner,Christopher Williams
3,L-10090,Andrew,Brown,Data Analytics Manager,Westbridge Legal,Legal,1001-5000,Denver,CO,United Sates,...,2026-05-24,None,25000000,45,75000,Jordan Lee,Demo Requested,Yes,Duplicate Suspected,Andrew Brown
4,L-10005,Emily,Garcia,Revops Manager,Metro Hospitality,Hospitality,5000+,Miami,FL,United Sates,...,2026-07-19,None,75000000,12,5000,Chris Brown,Phone Call,No,Referral From Partner,Emily Garcia


In [35]:
#Checking country unique values
df2['country'].unique()

array(['United Sates'], dtype=object)

In [36]:
#Checking for unique lead_ids 

print(df2['lead_id'].nunique())
print(df.shape)

123
(123, 27)


In [37]:
df2.isna().sum()

lead_id                       0
lead_first_name               0
lead_last_name                0
job_title                     0
company_name                  0
industry                      0
company_size                  0
city                          0
state                         0
country                       0
postal_code                   0
email                         3
phone                         4
lead_source                   0
campaign                      0
lead_status                   0
lead_score                    0
created_date                  0
first_contact_date           23
conversion_date             107
annual_revenue_usd            0
employees                     0
estimated_deal_value_usd      0
sales_rep                     0
last_activity                 0
marketing_opt_in             35
notes                        14
lead_full_name                0
dtype: int64

In [38]:
print(df2['job_title'].unique())

['Founder' 'Sales Director' 'Product Marketing Manager'
 'Data Analytics Manager' 'Revops Manager' 'Business Development Manager'
 'Demand Gen Manager' 'Ceo' 'Head Of Growth' 'Marketing Manager'
 'Vp Marketing' 'Director Of Marketing']


In [39]:
keywords = 'Founder|Ceo|Head|Vp|Director'

df2['is_decision_maker'] = df2['job_title'].str.contains(keywords, case=False, na=False)

df2[['job_title', 'is_decision_maker']].head()

,job_title,is_decision_maker
0,Founder,True
1,Sales Director,True
2,Product Marketing Manager,False
3,Data Analytics Manager,False
4,Revops Manager,False


In [40]:
df2.head()

,lead_id,lead_first_name,lead_last_name,job_title,company_name,industry,company_size,city,state,country,...,conversion_date,annual_revenue_usd,employees,estimated_deal_value_usd,sales_rep,last_activity,marketing_opt_in,notes,lead_full_name,is_decision_maker
0,L-10019,James,Moore,Founder,Silverstone Automotive,Automotive,501-1000,Columbus,OH,United Sates,...,None,500000000,3200,15000,Alex Kim,Phone Call,NaN,Downloaded Whitepaper,James Moore,True
1,L-10046,Andrew,Jackson,Sales Director,Lighthouse Biotech,Biotechnology,1-50,Cambridge,MA,United Sates,...,None,150000000,8500,40000,Chris Brown,Form Submission,NaN,Asked For Pricing,Andrew Jackson,True
2,L-10048,Christopher,Williams,Product Marketing Manager,Vertex Software,Technology,5000+,Austin,TX,United Sates,...,None,75000000,120,75000,Unassigned,Form Submission,NaN,Referral From Partner,Christopher Williams,False
3,L-10090,Andrew,Brown,Data Analytics Manager,Westbridge Legal,Legal,1001-5000,Denver,CO,United Sates,...,None,25000000,45,75000,Jordan Lee,Demo Requested,Yes,Duplicate Suspected,Andrew Brown,False
4,L-10005,Emily,Garcia,Revops Manager,Metro Hospitality,Hospitality,5000+,Miami,FL,United Sates,...,None,75000000,12,5000,Chris Brown,Phone Call,No,Referral From Partner,Emily Garcia,False


In [41]:
def assignment_flag(df, column_name):
    # Added () to .notna() and added .str.strip().str.lower() for safety
    is_assigned = (
        df2[column_name].notna()
        & (df2[column_name] != "")
        & (df2[column_name].astype(str).str.strip().str.lower() != "unassigned")
    )

    return np.where(is_assigned, "Assigned", "Unassigned")


# Apply the corrected function using your flexible column name variable
df2["is_assigned"] = assignment_flag(df2, "sales_rep")
df2.head()


,lead_id,lead_first_name,lead_last_name,job_title,company_name,industry,company_size,city,state,country,...,annual_revenue_usd,employees,estimated_deal_value_usd,sales_rep,last_activity,marketing_opt_in,notes,lead_full_name,is_decision_maker,is_assigned
0,L-10019,James,Moore,Founder,Silverstone Automotive,Automotive,501-1000,Columbus,OH,United Sates,...,500000000,3200,15000,Alex Kim,Phone Call,NaN,Downloaded Whitepaper,James Moore,True,Assigned
1,L-10046,Andrew,Jackson,Sales Director,Lighthouse Biotech,Biotechnology,1-50,Cambridge,MA,United Sates,...,150000000,8500,40000,Chris Brown,Form Submission,NaN,Asked For Pricing,Andrew Jackson,True,Assigned
2,L-10048,Christopher,Williams,Product Marketing Manager,Vertex Software,Technology,5000+,Austin,TX,United Sates,...,75000000,120,75000,Unassigned,Form Submission,NaN,Referral From Partner,Christopher Williams,False,Unassigned
3,L-10090,Andrew,Brown,Data Analytics Manager,Westbridge Legal,Legal,1001-5000,Denver,CO,United Sates,...,25000000,45,75000,Jordan Lee,Demo Requested,Yes,Duplicate Suspected,Andrew Brown,False,Assigned
4,L-10005,Emily,Garcia,Revops Manager,Metro Hospitality,Hospitality,5000+,Miami,FL,United Sates,...,75000000,12,5000,Chris Brown,Phone Call,No,Referral From Partner,Emily Garcia,False,Assigned


In [42]:
df2.to_pickle("lead_cleaning.pkl")

In [43]:
df2.to_csv('data_cleaning.csv')